# Fase B — Limpieza y normalización (parte 1: diagnóstico y calidad de datos)
**Dataset:** Dengue SIVIGILA 2022-2025 (581,071 registros, 20 columnas). **Objetivo:** `PAC_HOS` (hospitalizado sí/no, clasificación).
**Principio anti-leakage:** aquí solo se hace limpieza determinística, que no aprende nada de los datos. La imputación, el escalado y la codificación se ajustan después del split, solo con train.

## 1. Carga de datos
Se importan `pandas` y `numpy` y se lee el parquet consolidado del Proyecto 1 (`datos/procesados/dengue_consolidado.parquet`). La ruta es relativa (`../datos/...`) porque el notebook vive en la carpeta `entrega/`.

In [1]:
import numpy as np
import pandas as pd

RUTA = "../datos/procesados/dengue_consolidado.parquet"
df = pd.read_parquet(RUTA)

## 2. Diagnóstico inicial
### 2.1 Estructura y variable objetivo
Se revisa la forma del DataFrame, la distribución de `PAC_HOS`, las unidades de la edad (`UNI_MED`) y el resumen de `EDAD`.
- 581,071 filas y 20 columnas.
- `PAC_HOS` viene codificada 1 = Sí (38.4%) y 2 = No (61.6%): desbalance moderado.
- `UNI_MED`: 572,491 registros en años, 8,359 en meses, 213 en días y 8 en horas. Los 8,580 que no están en años son menores de 1 año; mezclados con años distorsionan la edad (media 23.17, máximo 122).

In [2]:
print(df.shape)
print(df["PAC_HOS"].value_counts(dropna=False, normalize=True).round(3))
print(df["UNI_MED"].value_counts(dropna=False))
print(df["EDAD"].describe())

(581071, 20)
PAC_HOS
2    0.616
1    0.384
Name: proportion, dtype: float64
UNI_MED
1    572491
2      8359
3       213
4         8
Name: count, dtype: int64
count    581071.000000
mean         23.165044
std          18.698874
min           1.000000
25%          10.000000
50%          17.000000
75%          32.000000
max         122.000000
Name: EDAD, dtype: float64


### 2.2 Calidad de columnas y verificación del Hallazgo 2 del EDA
Se arma una tabla con tipo, % de nulos, valores únicos y un ejemplo por columna, y se cuentan los duplicados exactos. Luego se calcula la tasa real de hospitalización por grupo de edad (las edades no expresadas en años se tratan como menores de 1 año).
- No hay nulos reales (0% en todas las columnas), pero sí nulos disfrazados de texto en `estrato`.
- Hay 20,454 filas duplicadas exactas.
- Hospitalización por edad: <1 año 48.6%, 1-4 36.5%, 5-12 39.8%, 13-17 44.4%, 18-29 37.5%, 30-49 30.9%, 50-69 35.7%, 70+ 47.6%.

**Conclusión:** la relación tiene forma de U (más hospitalización en bebés y adultos mayores). El Hallazgo 2 del EDA decía ">50% en todos los rangos, con pico de ~70% en 30-49 años". Ese 70% coincide con el complemento de la tasa real (100 − 30.9 = 69.1%), lo que sugiere que se graficó `PAC_HOS == 2` (No hospitalizado) en vez de `== 1`. Hay que corregir el Hallazgo 2 y revisar los gráficos del EDA que usen `PAC_HOS`.

In [3]:
# 1) Resumen de las 20 columnas
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(),
    "ejemplo": df.iloc[0],
})
print(resumen.to_string())
print("Duplicados exactos:", df.duplicated().sum())

# 2) Verifica el Hallazgo 2: tasa real de hospitalización (PAC_HOS == 1 es "Sí")
# Edad provisional: si UNI_MED no es años, es menor de 1 año -> 0
anios = df["EDAD"].where(df["UNI_MED"] == 1, 0)
grupo = pd.cut(anios, [-1, 0, 4, 12, 17, 29, 49, 69, 130],
               labels=["<1", "1-4", "5-12", "13-17", "18-29", "30-49", "50-69", "70+"])
print((df["PAC_HOS"] == 1).groupby(grupo, observed=True).mean().round(3))

                           tipo  nulos_%  unicos                     ejemplo
SEMANA                    int64      0.0      53                          11
ANO                       int64      0.0       4                        2022
EDAD                      int64      0.0     108                           6
UNI_MED                   int64      0.0       4                           1
SEXO                        str      0.0       2                           M
COD_DPTO_O                int64      0.0      33                          44
COD_MUN_O                 int64      0.0     549                           1
AREA                      int64      0.0       3                           1
TIP_SS                      str      0.0       6                           S
PER_ETN                   int64      0.0       6                           6
estrato                     str      0.0       8                  2         
GP_DESPLAZ                int64      0.0       2                           2

### 2.3 Variables categóricas y valores centinela
Se imprime la frecuencia de cada valor de las variables categóricas y binarias, y se cuentan los registros con edad mayor a 100 años.
- `estrato`: valores con espacios al final, 20,422 vacíos (3.5%) y 3,082 "SIN DATO" (0.5%). Son 23,504 nulos disfrazados (4.0%).
- `SEXO` (2 valores) y `TIP_SS` (6 valores) están limpias: sin mayúsculas ni códigos mezclados.
- `GP_DESPLAZ`, `GP_MIGRANT` y `GP_GESTAN` son binarias 1 = Sí / 2 = No, con 99.6%, 98.9% y 99.5% en "No".
- `AJUSTE` (0/3/5) y `nom_est_f_caso` parecen campos posteriores a la notificación (se justifica en la sección 5).
- 21 registros tienen edad mayor a 100 años.

In [4]:
cols_cat = ["SEXO", "TIP_SS", "estrato", "AJUSTE", "nom_est_f_caso",
            "AREA", "PER_ETN", "GP_DESPLAZ", "GP_MIGRANT", "GP_GESTAN", "TIP_CAS"]
for c in cols_cat:
    print(f"\n== {c} ==")
    for valor, n in df[c].value_counts(dropna=False).items():
        print(f"{valor!r:32} {n:>8}  ({n/len(df):.1%})")

print("\nEDAD > 100 (en años):", ((df["EDAD"] > 100) & (df["UNI_MED"] == 1)).sum())


== SEXO ==
'M'                                295551  (50.9%)
'F'                                285520  (49.1%)

== TIP_SS ==
'S'                                293885  (50.6%)
'C'                                251661  (43.3%)
'P'                                 24074  (4.1%)
'N'                                  6816  (1.2%)
'E'                                  2774  (0.5%)
'I'                                  1861  (0.3%)

== estrato ==
'1         '                       261574  (45.0%)
'2         '                       202288  (34.8%)
'3         '                        72868  (12.5%)
'          '                        20422  (3.5%)
'4         '                        14617  (2.5%)
'5         '                         4378  (0.8%)
'SIN DATO  '                         3082  (0.5%)
'6         '                         1842  (0.3%)

== AJUSTE ==
'0'                                329933  (56.8%)
'3'                                225968  (38.9%)
'5'                                 

## 3. Función `limpiar_basico`
Limpieza determinística y reutilizable: no calcula medias ni modas, así que no hay riesgo de data leakage. Devuelve el DataFrame limpio antes de codificar, para que la Parte 2 (split, encoding y escalado) continúe desde ahí.

| Columna | Problema detectado | Decisión | Justificación |
|---|---|---|---|
| Nombres de columnas | Mayúsculas y minúsculas mezcladas | Pasar todo a mayúsculas | Nombres consistentes en todo el pipeline |
| Texto (`SEXO`, `TIP_SS`, departamento, municipio) | Espacios sobrantes | `strip()` | Evita categorías duplicadas |
| `estrato` | Vacíos y "SIN DATO" (4.0%) | Convertir a NaN y a número (1-6) | Son nulos reales; se imputan después del split |
| `EDAD` + `UNI_MED` | 8,580 registros en meses, días u horas | Convertir todo a años en `EDAD_ANIOS` | Evita mezclar unidades |
| `EDAD_ANIOS` | 21 registros mayores de 100 años | Pasar a NaN (se imputa con mediana tras el split) | Valores implausibles |
| `GP_*` | Codificadas 1 = Sí / 2 = No | Recodificar a 1/0 | Formato binario estándar |
| `PAC_HOS` | 1 = Sí / 2 = No | Recodificar: 1 = hospitalizado, 0 = no | Objetivo en formato estándar |
| `AJUSTE` | Campo posterior a la notificación | Excluir | Evita fuga de información del desenlace |
| `nom_est_f_caso` | Clasificación final del caso | Excluir | Ídem (ver sección 5) |

In [5]:
def limpiar_basico(df, quitar_duplicados=False, conservar_estado_final=False):
    """Limpieza determinística: no aprende nada de los datos (sin imputar ni escalar).
    La imputación y el escalado se ajustan después del split, solo con train.
    conservar_estado_final=True mantiene NOM_EST_F_CASO (solo para análisis de evidencia)."""
    df = df.copy()

    # 0) Duplicados exactos (se conservan por defecto; ver sección 5)
    if quitar_duplicados:
        df = df.drop_duplicates().reset_index(drop=True)

    # 1) Nombres de columnas consistentes
    df.columns = [c.upper() for c in df.columns]

    # 2) Texto: quitar espacios sobrantes
    for c in ["SEXO", "TIP_SS", "NOM_EST_F_CASO", "DEPARTAMENTO_OCURRENCIA", "MUNICIPIO_OCURRENCIA"]:
        df[c] = df[c].astype("string").str.strip()

    # 3) ESTRATO: vacío y "SIN DATO" son nulos disfrazados -> NaN; luego a número 1-6
    est = df["ESTRATO"].astype("string").str.strip()
    est = est.replace({"": pd.NA, "SIN DATO": pd.NA})
    df["ESTRATO"] = pd.to_numeric(est, errors="coerce").astype("float64")

    # 4) EDAD a años usando UNI_MED (1=años, 2=meses, 3=días, 4=horas)
    factor = {1: 1, 2: 1 / 12, 3: 1 / 365, 4: 1 / 8760}
    df["EDAD_ANIOS"] = df["EDAD"] * df["UNI_MED"].map(factor)
    df.loc[df["EDAD_ANIOS"] > 100, "EDAD_ANIOS"] = np.nan   # 21 edades implausibles
    df = df.drop(columns=["EDAD", "UNI_MED"])

    # 5) Banderas 1=Sí / 2=No -> 0/1
    for c in ["GP_DESPLAZ", "GP_MIGRANT", "GP_GESTAN"]:
        df[c] = (df[c] == 1).astype("int8")

    # 6) Objetivo: PAC_HOS queda 1 = hospitalizado, 0 = no
    df["PAC_HOS"] = (df["PAC_HOS"] == 1).astype("int8")

    # 7) Se excluyen campos posteriores a la notificación
    cols_excluir = ["AJUSTE"] if conservar_estado_final else ["AJUSTE", "NOM_EST_F_CASO"]
    df = df.drop(columns=cols_excluir)
    return df

## 4. Validación de la función
Se aplica `limpiar_basico` y se verifica el resultado. Resultado esperado:
- Forma `(581071, 17)`: se quitan `EDAD`, `UNI_MED`, `AJUSTE` y `nom_est_f_caso`, y se agrega `EDAD_ANIOS`.
- Nulos solo en `ESTRATO` (23,504) y `EDAD_ANIOS` (21).
- Media de `PAC_HOS` = 0.384, igual que antes de limpiar.
- `EDAD_ANIOS` con máximo 100.

In [6]:
df_limpio = limpiar_basico(df)
print(df_limpio.shape)
print(df_limpio.isna().sum()[lambda s: s > 0])
print(round(df_limpio["PAC_HOS"].mean(), 3))
print(df_limpio["EDAD_ANIOS"].describe().round(2))

(581071, 17)
ESTRATO       23504
EDAD_ANIOS       21
dtype: int64
0.384
count    581050.00
mean         23.06
std          18.79
min           0.00
25%           9.00
50%          17.00
75%          32.00
max         100.00
Name: EDAD_ANIOS, dtype: float64


## 5. Evidencia para las decisiones
Esta celda no limpia nada: genera la evidencia que justifica dos decisiones. Usa una copia temporal (`df_ev`) que conserva `nom_est_f_caso` solo para este análisis.
- **A) Duplicados:** frecuencia de filas repetidas y su relación con el volumen de casos del municipio.
- **B) `TIP_CAS` vs `nom_est_f_caso`:** cruce de ambas variables y tasa de hospitalización por categoría.

In [7]:
# Versión que conserva nom_est_f_caso solo para justificar su exclusión
df_ev = limpiar_basico(df, conservar_estado_final=True)

# A) Duplicados: ¿su frecuencia crece con el volumen del municipio?
es_dup = df_ev.duplicated()
copias = df_ev.value_counts(dropna=False)
print("Filas repetidas (extra):", es_dup.sum())
print("Copias por fila repetida:\n", copias[copias > 1].describe().round(2))

n_mun = df_ev["MUNICIPIO_OCURRENCIA"].value_counts()
volumen = df_ev["MUNICIPIO_OCURRENCIA"].map(n_mun)
print("\nTasa de duplicados según casos totales del municipio:")
print(es_dup.groupby(pd.cut(volumen, [0, 100, 1000, 10000, 1e6]), observed=True).mean().round(3))

# B) TIP_CAS vs NOM_EST_F_CASO
print("\n", pd.crosstab(df_ev["TIP_CAS"], df_ev["NOM_EST_F_CASO"]))
print("\nTasa de hospitalización por TIP_CAS:")
print(df_ev.groupby("TIP_CAS")["PAC_HOS"].mean().round(3))
print("\nTasa de hospitalización por NOM_EST_F_CASO:")
print(df_ev.groupby("NOM_EST_F_CASO")["PAC_HOS"].mean().round(3))

Filas repetidas (extra): 21187
Copias por fila repetida:
 count    17883.00
mean         2.18
std          0.52
min          2.00
25%          2.00
50%          2.00
75%          2.00
max          7.00
Name: count, dtype: float64

Tasa de duplicados según casos totales del municipio:
MUNICIPIO_OCURRENCIA
(0.0, 100.0]            0.002
(100.0, 1000.0]         0.007
(1000.0, 10000.0]       0.024
(10000.0, 1000000.0]    0.087
dtype: float64

 NOM_EST_F_CASO  Confirmado por Nexo Epidemiológico  \
TIP_CAS                                              
2                                            24378   
3                                              291   
5                                             2516   

NOM_EST_F_CASO  Confirmado por laboratorio  Probable  
TIP_CAS                                               
2                                   197359    129502  
3                                   226715         0  
5                                      310         0  

Tasa de ho

### Decisión sobre duplicados
Se detectaron 20,454 filas exactamente iguales en el dataset original (21,187 tras la limpieza básica, antes de excluir `nom_est_f_caso`). Como el dataset no tiene identificador de paciente, se decidió **conservarlas**: la tasa de duplicados crece con el volumen de casos del municipio (0.2% en municipios de menos de 100 casos frente a 8.7% en los de más de 10,000) y casi todas las filas repetidas aparecen solo dos veces. Ese patrón es el esperable cuando pacientes distintos comparten el mismo perfil, y no el de registros cargados por error. Limitación: algunas filas idénticas pueden quedar en train y en test; con ~3.6% de filas, el efecto esperado es pequeño.

### Variables excluidas por ser posteriores a la notificación
Se excluyen `AJUSTE` y `nom_est_f_caso`. Esta última es la clasificación final del caso: entre los casos que se notificaron como "probables", los que luego quedaron confirmados se hospitalizan mucho más (~47%) que los que siguen probables (16%), lo que indica que el estado final incorpora información del desenlace. Se conserva `TIP_CAS`, la clasificación al notificar.

In [8]:
# 1) Resumen de las 20 columnas
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(),
    "ejemplo": df.iloc[0],
})
print(resumen.to_string())
print("Duplicados exactos:", df.duplicated().sum())

# 2) Verifica el Hallazgo 2: tasa real de hospitalización (PAC_HOS == 1 es "Sí")
# Edad provisional: si UNI_MED no es años, es menor de 1 año -> 0
anios = df["EDAD"].where(df["UNI_MED"] == 1, 0)
grupo = pd.cut(anios, [-1, 0, 4, 12, 17, 29, 49, 69, 130],
               labels=["<1", "1-4", "5-12", "13-17", "18-29", "30-49", "50-69", "70+"])
print((df["PAC_HOS"] == 1).groupby(grupo, observed=True).mean().round(3))

                           tipo  nulos_%  unicos                     ejemplo
SEMANA                    int64      0.0      53                          11
ANO                       int64      0.0       4                        2022
EDAD                      int64      0.0     108                           6
UNI_MED                   int64      0.0       4                           1
SEXO                        str      0.0       2                           M
COD_DPTO_O                int64      0.0      33                          44
COD_MUN_O                 int64      0.0     549                           1
AREA                      int64      0.0       3                           1
TIP_SS                      str      0.0       6                           S
PER_ETN                   int64      0.0       6                           6
estrato                     str      0.0       8                  2         
GP_DESPLAZ                int64      0.0       2                           2

In [9]:
import pandas as pd
from pathlib import Path

# Lista los archivos de datos del repo
for p in Path("..").rglob("*"):
    if p.suffix in {".parquet", ".csv"} and ".git" not in p.parts:
        print(p, round(p.stat().st_size / 1e6, 1), "MB")

..\datos\procesados\dengue_consolidado.parquet 4.5 MB


In [10]:
cols_cat = ["SEXO", "TIP_SS", "estrato", "AJUSTE", "nom_est_f_caso",
            "AREA", "PER_ETN", "GP_DESPLAZ", "GP_MIGRANT", "GP_GESTAN", "TIP_CAS"]
for c in cols_cat:
    print(f"\n== {c} ==")
    for valor, n in df[c].value_counts(dropna=False).items():
        print(f"{valor!r:32} {n:>8}  ({n/len(df):.1%})")

print("\nEDAD > 100 (en años):", ((df["EDAD"] > 100) & (df["UNI_MED"] == 1)).sum())


== SEXO ==
'M'                                295551  (50.9%)
'F'                                285520  (49.1%)

== TIP_SS ==
'S'                                293885  (50.6%)
'C'                                251661  (43.3%)
'P'                                 24074  (4.1%)
'N'                                  6816  (1.2%)
'E'                                  2774  (0.5%)
'I'                                  1861  (0.3%)

== estrato ==
'1         '                       261574  (45.0%)
'2         '                       202288  (34.8%)
'3         '                        72868  (12.5%)
'          '                        20422  (3.5%)
'4         '                        14617  (2.5%)
'5         '                         4378  (0.8%)
'SIN DATO  '                         3082  (0.5%)
'6         '                         1842  (0.3%)

== AJUSTE ==
'0'                                329933  (56.8%)
'3'                                225968  (38.9%)
'5'                                 

In [11]:
import numpy as np
import pandas as pd

def limpiar_basico(df, quitar_duplicados=False):
    """Limpieza determinística: no aprende nada de los datos (sin imputar ni escalar).
    La imputación y el escalado se ajustan después del split, solo con train."""
    df = df.copy()

    # 0) Duplicados exactos (decisión pendiente; se aplica antes de tocar columnas)
    if quitar_duplicados:
        df = df.drop_duplicates().reset_index(drop=True)

    # 1) Nombres de columnas consistentes
    df.columns = [c.upper() for c in df.columns]

    # 2) Texto: quitar espacios sobrantes
    for c in ["SEXO", "TIP_SS", "NOM_EST_F_CASO", "DEPARTAMENTO_OCURRENCIA", "MUNICIPIO_OCURRENCIA"]:
        df[c] = df[c].astype("string").str.strip()

    # 3) ESTRATO: vacío y "SIN DATO" son nulos disfrazados -> NaN; luego a número 1-6
    est = df["ESTRATO"].astype("string").str.strip()
    est = est.replace({"": pd.NA, "SIN DATO": pd.NA})
    df["ESTRATO"] = pd.to_numeric(est, errors="coerce").astype("float64")

    # 4) EDAD a años usando UNI_MED (1=años, 2=meses, 3=días, 4=horas)
    factor = {1: 1, 2: 1 / 12, 3: 1 / 365, 4: 1 / 8760}
    df["EDAD_ANIOS"] = df["EDAD"] * df["UNI_MED"].map(factor)
    df.loc[df["EDAD_ANIOS"] > 100, "EDAD_ANIOS"] = np.nan   # 21 edades implausibles
    df = df.drop(columns=["EDAD", "UNI_MED"])

    # 5) Banderas 1=Sí / 2=No -> 0/1
    for c in ["GP_DESPLAZ", "GP_MIGRANT", "GP_GESTAN"]:
        df[c] = (df[c] == 1).astype("int8")

    # 6) Objetivo: PAC_HOS queda 1 = hospitalizado, 0 = no
    df["PAC_HOS"] = (df["PAC_HOS"] == 1).astype("int8")

    # 7) Se excluyen campos posteriores a la notificación
    df = df.drop(columns=["AJUSTE", "NOM_EST_F_CASO"])
    return df

In [12]:
df_limpio = limpiar_basico(df)
print(df_limpio.shape)

(581071, 17)


### Decisión sobre duplicados
Se detectaron 20,454 filas exactamente iguales en el dataset original (21,187 tras la limpieza básica). Como el dataset no tiene identificador de paciente, se decidió **conservarlas**: la tasa de duplicados crece con el volumen de casos del municipio (0.2% en municipios de menos de 100 casos frente a 8.7% en los de más de 10,000) y casi todas las filas repetidas aparecen solo dos veces. Ese patrón es el esperable cuando pacientes distintos comparten el mismo perfil, y no el de registros cargados por error. Limitación: algunas filas idénticas pueden quedar en train y en test; con ~3.6% de filas, el efecto esperado es pequeño.

### Variables excluidas por ser posteriores a la notificación
Se excluyen `AJUSTE` y `nom_est_f_caso`. Esta última es la clasificación final del caso: entre los casos que se notificaron como "probables", los que luego quedaron confirmados se hospitalizan mucho más (~47%) que los que siguen probables (16%), lo que indica que el estado final incorpora información del desenlace. Se conserva `TIP_CAS`, la clasificación al notificar.

## 6. Outliers de `EDAD_ANIOS` (método IQR)
Se usa el método IQR y no el Z-score porque `EDAD` tiene sesgo positivo (~1.3) y el Z-score supone una distribución aproximadamente normal. Además, la media y la desviación se distorsionan justo por los valores extremos que se quieren detectar, mientras que los cuartiles no.

**Regla:** son atípicos los valores fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

Esta celda es solo diagnóstico: no modifica los datos. Se comparan los atípicos con el resto para decidir si son errores o casos reales. Como no se elimina ni se recorta nada, no hay data leakage; si más adelante se recortaran valores, los límites se calcularían solo con train.


In [13]:
# Outliers de EDAD_ANIOS con IQR (diagnóstico: no se modifica nada)
edad = df_limpio["EDAD_ANIOS"]
q1, q3 = edad.quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}")
print(f"Límites: [{lim_inf:.2f}, {lim_sup:.2f}]")

es_out = (edad < lim_inf) | (edad > lim_sup)
print("\nOutliers:", es_out.sum(), f"({es_out.sum() / edad.notna().sum():.2%} de los registros con edad)")
print("Por debajo del límite inferior:", (edad < lim_inf).sum())

# ¿Son errores o casos reales? Se compara su perfil con el resto
print("\nEdad de los outliers:")
print(edad[es_out].describe().round(1))
print("\nTasa de hospitalización:")
print("  outliers:", round(df_limpio.loc[es_out, "PAC_HOS"].mean(), 3))
print("  resto   :", round(df_limpio.loc[~es_out & edad.notna(), "PAC_HOS"].mean(), 3))

Q1=9.00  Q3=32.00  IQR=23.00
Límites: [-25.50, 66.50]

Outliers: 22816 (3.93% de los registros con edad)
Por debajo del límite inferior: 0

Edad de los outliers:
count    22816.0
mean        75.7
std          7.0
min         67.0
25%         70.0
50%         74.0
75%         80.0
max        100.0
Name: EDAD_ANIOS, dtype: float64

Tasa de hospitalización:
  outliers: 0.462
  resto   : 0.381


### Resultado y decisión sobre los outliers
Con Q1 = 9, Q3 = 32 e IQR = 23, los límites son [−25.5, 66.5]. No hay atípicos por debajo (la edad no puede ser negativa) y hay **22,816 por encima (3.93% de los registros con edad)**: personas de 67 a 100 años, con mediana de 74.

**Decisión: conservarlos.**
- **Son valores reales, no errores.** Las edades de 67 a 100 años son plausibles y corresponden a la cola natural de la población adulta mayor. Los únicos valores descartados por implausibles son los 21 registros mayores de 100 años, que ya pasaron a NaN en `limpiar_basico`.
- **Aportan información al modelo.** Su tasa de hospitalización es 46.2%, frente a 38.1% en el resto. Eliminarlos quitaría justo al grupo de mayor riesgo y sesgaría el modelo.
- **La cola larga se controla al escalar, no eliminando.** En la Parte 2 se usará `RobustScaler` (basado en mediana e IQR), que no se distorsiona con estos valores.

## 7. Split train/test (primero, antes de imputar o escalar)
Se parte el dataset en 80% entrenamiento y 20% prueba **antes** de cualquier paso que aprenda de los datos (imputación, escalado). Así el conjunto de prueba queda realmente "nunca visto" y se evita el data leakage que prohíbe el enunciado.
- **Estratificado por `PAC_HOS`:** train y test mantienen la misma proporción de hospitalizados (38.4%).
- **Semilla fija (`random_state=42`):** el split es reproducible. Las Personas 4 y 5 usarán exactamente esta partición mediante la columna `split` del parquet.
- Las filas duplicadas se conservan (sección 5), por lo que algunas filas idénticas pueden caer en ambos conjuntos.

In [14]:
from sklearn.model_selection import train_test_split

SEED = 42  # semilla fija: el mismo split lo usarán las Personas 4 y 5
assert df_limpio.index.is_unique

pos = np.arange(len(df_limpio))
pos_train, pos_test = train_test_split(
    pos, test_size=0.20, stratify=df_limpio["PAC_HOS"], random_state=SEED
)
es_test = np.zeros(len(df_limpio), dtype=bool)
es_test[pos_test] = True

df_split = df_limpio.copy()
df_split["split"] = np.where(es_test, "test", "train")

train = df_split[df_split["split"] == "train"].drop(columns="split")
test = df_split[df_split["split"] == "test"].drop(columns="split")

print("Filas train:", len(train), "| test:", len(test))
print("\nProporción de hospitalizados:")
print("  total:", round(df_split["PAC_HOS"].mean(), 4))
print("  train:", round(train["PAC_HOS"].mean(), 4))
print("  test :", round(test["PAC_HOS"].mean(), 4))
print("\nNulos por conjunto:")
print(pd.DataFrame({"train": train.isna().sum(), "test": test.isna().sum()}).query("train > 0 or test > 0"))

Filas train: 464856 | test: 116215

Proporción de hospitalizados:
  total: 0.3844
  train: 0.3844
  test : 0.3844

Nulos por conjunto:
            train  test
ESTRATO     18809  4695
EDAD_ANIOS     17     4


## 8. Imputación de nulos (ajustada solo con train)
Solo quedan dos columnas con nulos, y cada una se trata con una estrategia distinta según su tipo:

| Columna | Nulos | Tipo | Estrategia |
|---|---|---|---|
| `EDAD_ANIOS` | 21 en total (edades > 100 años) | Numérica asimétrica | **Mediana** de train: robusta al sesgo y a valores extremos; con solo 21 casos el efecto es mínimo |
| `ESTRATO` | 23,504 (4.0%) | Ordinal (1-6) | **Moda vs KNN**: se comparan con un experimento y se elige la mejor |

**Experimento para `ESTRATO`:** con una submuestra de train (por costo computacional, KNN con ~465 mil filas es lento) se oculta el 10% de los estratos conocidos, se imputan con cada estrategia y se compara con el valor real. Se miden la exactitud (% de aciertos exactos) y el error absoluto medio (cuántos estratos se equivoca en promedio). KNN usa tipo de afiliación (`TIP_SS`), área, departamento, pertenencia étnica y edad.

In [15]:
from sklearn.impute import KNNImputer

cols_knn = ["TIP_SS", "AREA", "COD_DPTO_O", "PER_ETN"]

# Submuestra de train con ESTRATO conocido (KNN es costoso con 465 mil filas)
sub = train[train["ESTRATO"].notna()].sample(n=40000, random_state=SEED).copy()

# Se oculta el 10% de los estratos conocidos para medir qué tan bien se recuperan
ocultos = sub.sample(frac=0.10, random_state=SEED).index
real = sub.loc[ocultos, "ESTRATO"].copy()
sub.loc[ocultos, "ESTRATO"] = np.nan

# Estrategia 1: moda (calculada solo con los estratos no ocultos)
moda = sub["ESTRATO"].mode()[0]
pred_moda = pd.Series(moda, index=ocultos)

# Estrategia 2: KNN sobre variables sociodemográficas (one-hot + edad)
X = pd.get_dummies(sub[cols_knn].astype(str), dtype=float)
X["EDAD_ANIOS"] = sub["EDAD_ANIOS"].fillna(train["EDAD_ANIOS"].median()) / 100
X["ESTRATO"] = sub["ESTRATO"]
imputado = KNNImputer(n_neighbors=5).fit_transform(X)
pred_knn = pd.Series(np.rint(imputado[:, X.columns.get_loc("ESTRATO")]), index=sub.index).loc[ocultos]

resultado = pd.DataFrame({
    "exactitud": [(pred_moda == real).mean(), (pred_knn == real).mean()],
    "error_abs_medio": [(pred_moda - real).abs().mean(), (pred_knn - real).abs().mean()],
}, index=["Moda", "KNN (k=5)"]).round(3)
print("Moda de ESTRATO en train:", moda)
print(resultado)

Moda de ESTRATO en train: 1.0
           exactitud  error_abs_medio
Moda           0.481            0.744
KNN (k=5)      0.560            0.529


### Resultado y decisión: imputación de `ESTRATO`
| Estrategia | Exactitud | Error absoluto medio |
|---|---|---|
| Moda (siempre estrato 1) | 48.1% | 0.744 |
| KNN (k=5) | 56.0% | 0.529 |

**Decisión: KNN para `ESTRATO`.** Acierta el estrato exacto casi 8 puntos más que la moda (56.0% vs 48.1%) y, cuando se equivoca, se desvía menos (error medio de 0.53 estratos frente a 0.74, una reducción de ~29%). La moda asignaría estrato 1 a todos los vacíos y distorsionaría la distribución de los demás estratos; KNN aprovecha que el estrato se relaciona con el tipo de afiliación, el área y el departamento. El promedio de los 5 vecinos se redondea al estrato entero más cercano, porque el estrato es ordinal.

**Limitaciones:** el experimento supone que los estratos faltantes se parecen a los conocidos (nulos aleatorios), algo que no se puede comprobar, y una exactitud de 56% es moderada, así que el `ESTRATO` imputado conserva incertidumbre.

**Implementación sin leakage:** los vecinos (donantes) salen solo de una submuestra de 40,000 filas de train con estrato conocido. Train y test se imputan con esos mismos donantes, y la mediana de `EDAD_ANIOS` también se calcula solo con train.

**Verificación:** tras imputar no quedan nulos en train ni en test, y la distribución de `ESTRATO` casi no cambia (p. ej. estrato 1: 46.9% → 46.2%), es decir, la imputación no distorsiona la variable.

In [16]:
from sklearn.neighbors import NearestNeighbors

mediana_edad = train["EDAD_ANIOS"].median()   # solo con train

def matriz_knn(d, columnas_ref=None):
    """Variables para buscar vecinos: one-hot de cols_knn + edad escalada."""
    X = pd.get_dummies(d[cols_knn].astype(str), dtype=float)
    X["EDAD_ANIOS"] = d["EDAD_ANIOS"].fillna(mediana_edad) / 100
    if columnas_ref is not None:
        X = X.reindex(columns=columnas_ref, fill_value=0.0)
    return X

# Donantes: submuestra de train con ESTRATO conocido
donantes = train[train["ESTRATO"].notna()].sample(n=40000, random_state=SEED)
X_don = matriz_knn(donantes)
knn = NearestNeighbors(n_neighbors=5, algorithm="brute").fit(X_don)

def imputar(d):
    """ESTRATO: promedio redondeado de los 5 vecinos. EDAD_ANIOS: mediana de train."""
    d = d.copy()
    falta = d["ESTRATO"].isna()
    vecinos = knn.kneighbors(matriz_knn(d[falta], X_don.columns), return_distance=False)
    d.loc[falta, "ESTRATO"] = np.rint(donantes["ESTRATO"].to_numpy()[vecinos].mean(axis=1))
    d["EDAD_ANIOS"] = d["EDAD_ANIOS"].fillna(mediana_edad)
    return d

train_imp = imputar(train)
test_imp = imputar(test)

print("Mediana de EDAD_ANIOS (train):", mediana_edad)
print("Nulos restantes -> train:", train_imp.isna().sum().sum(), "| test:", test_imp.isna().sum().sum())
print("\nDistribución de ESTRATO en train (%):")
print(pd.DataFrame({
    "antes": train["ESTRATO"].value_counts(normalize=True).sort_index() * 100,
    "después": train_imp["ESTRATO"].value_counts(normalize=True).sort_index() * 100,
}).round(1))

Mediana de EDAD_ANIOS (train): 17.0
Nulos restantes -> train: 0 | test: 0

Distribución de ESTRATO en train (%):
         antes  después
ESTRATO                
1.0       46.9     46.2
2.0       36.3     37.0
3.0       13.1     13.2
4.0        2.6      2.5
5.0        0.8      0.7
6.0        0.3      0.3


## 9. Variables derivadas y sesgo
Antes de codificar y escalar se preparan dos variables y se mide el sesgo (asimetría) de las numéricas continuas usando solo train:
- **`SEMANA` cíclica:** la semana 52 está pegada a la 1, pero numéricamente están lejos. Se representa con seno y coseno para que el modelo "vea" el ciclo anual.
- **Municipio:** tiene 974 niveles (demasiados para One-Hot, que crearía ~970 columnas casi vacías) y `COD_MUN_O` no sirve solo porque su código se repite entre departamentos. Se reemplaza por una medida numérica: el número de casos del municipio en train (`MUN_CASOS`) y su logaritmo (`MUN_LOGCASOS`), que resumen su tamaño. Los municipios que no aparecen en train reciben 0. El departamento ya se codifica aparte.
- **Sesgo:** se compara la asimetría de `EDAD_ANIOS` y `MUN_CASOS`, crudas y transformadas (raíz, log y Yeo-Johnson), para decidir si hace falta transformarlas. Una asimetría cercana a 0 indica distribución simétrica; por encima de ~1 se considera sesgo fuerte.

In [17]:
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer

# Conteo de casos por municipio, calculado SOLO con train
conteo_mun = train_imp["MUNICIPIO_OCURRENCIA"].value_counts()

def derivar(d):
    """Agrega SEMANA cíclica (seno/coseno) y las medidas de tamaño del municipio."""
    d = d.copy()
    d["SEMANA_SIN"] = np.sin(2 * np.pi * d["SEMANA"] / 52)
    d["SEMANA_COS"] = np.cos(2 * np.pi * d["SEMANA"] / 52)
    d["MUN_CASOS"] = d["MUNICIPIO_OCURRENCIA"].map(conteo_mun).fillna(0)
    d["MUN_LOGCASOS"] = np.log1p(d["MUN_CASOS"])
    return d

train_d = derivar(train_imp)
test_d = derivar(test_imp)
print("Filas de test con municipio no visto en train:",
      (~test_imp["MUNICIPIO_OCURRENCIA"].isin(conteo_mun.index)).sum())

def sesgos(x):
    yj = PowerTransformer(method="yeo-johnson").fit_transform(x.reshape(-1, 1)).ravel()
    return {"crudo": skew(x), "raíz": skew(np.sqrt(x)), "log1p": skew(np.log1p(x)), "Yeo-Johnson": skew(yj)}

print("\nAsimetría (train):")
print(pd.DataFrame({
    "EDAD_ANIOS": sesgos(train_d["EDAD_ANIOS"].to_numpy()),
    "MUN_CASOS": sesgos(train_d["MUN_CASOS"].to_numpy()),
}).round(2))

Filas de test con municipio no visto en train: 18

Asimetría (train):
             EDAD_ANIOS  MUN_CASOS
crudo              1.25       2.17
raíz               0.46       1.25
log1p             -0.46      -0.17
Yeo-Johnson       -0.02      -0.02


### Resultado y decisiones de transformación
| Variable | Crudo | Raíz | log1p | Yeo-Johnson |
|---|---|---|---|---|
| `EDAD_ANIOS` | 1.25 | 0.46 | −0.46 | −0.02 |
| `MUN_CASOS` | 2.17 | 1.25 | −0.17 | −0.02 |

- **`EDAD_ANIOS` → Yeo-Johnson.** Su sesgo crudo (1.25) es fuerte. La raíz lo deja todavía asimétrico (0.46) y el log1p lo sobrecorrige hacia el otro lado (−0.46); Yeo-Johnson lo deja casi simétrico (−0.02). Se prefiere a Box-Cox porque Box-Cox exige valores estrictamente positivos y Yeo-Johnson admite cualquier valor. Se ajusta solo con train y luego se aplica `RobustScaler`, para que los adultos mayores conservados (sección 6) no dominen la escala.
- **`MUN_CASOS` → log1p.** Pasa de 2.17 a −0.17, suficiente para una distribución casi simétrica y más fácil de explicar que Yeo-Johnson. Después se estandariza (Z-score), porque tras el log no quedan valores extremos.
- **Sin transformar:** `ESTRATO` y `ANO` (ordinales acotados), `SEMANA_SIN` y `SEMANA_COS` (ya están en [−1, 1]) y las banderas 0/1.
- La transformación conserva el orden de la edad, así que no afecta a los modelos de árboles; ayuda sobre todo a la regresión logística.
- Hay 18 filas de test con un municipio no visto en train: reciben 0 casos, como cualquier municipio sin historial.

## 10. Codificación y escalado (ColumnTransformer ajustado solo con train)
Se construye un `ColumnTransformer` de scikit-learn que aplica a cada variable el tratamiento que le corresponde. Se **ajusta (`fit`) solo con train** y luego se aplica a train y test, de modo que ningún parámetro (categorías, medias, medianas, mínimos) se calcula con el conjunto de prueba.

| Variable(s) | Tratamiento | Justificación |
|---|---|---|
| `SEXO`, `TIP_SS`, `AREA`, `PER_ETN`, `TIP_CAS`, `COD_DPTO_O`, `GRUPO_EDAD` | One-Hot con `drop='first'` | Son nominales sin orden. `COD_DPTO_O` es un código, no una cantidad, por eso no se trata como numérico. Quitar la primera categoría evita la colinealidad perfecta entre dummies (importante en regresión logística) |
| `ESTRATO` | Ordinal (1-6 → 0-5) + Min-Max | Tiene orden natural. Min-Max lo deja en [0, 1], la misma escala que las dummies |
| `ANO` | Min-Max | Ordinal acotado (2022-2025) |
| `EDAD_ANIOS` | Yeo-Johnson + `RobustScaler` | Sesgo fuerte (1.25) y adultos mayores conservados como valores extremos (secciones 6 y 9) |
| `MUN_LOGCASOS` | Z-score (`StandardScaler`) | Tras el log es casi simétrica y sin extremos |
| `SEMANA_SIN`, `SEMANA_COS`, `GP_*` | Sin escalar | Ya están en [−1, 1] o son 0/1 |

**Variable nueva `GRUPO_EDAD`:** la sección 2.2 mostró que la hospitalización tiene forma de U según la edad (más alta en bebés y adultos mayores). Una regresión logística con la edad como un solo número no puede captar una U, así que se agregan grupos fijos de edad (<1, 1-4, 5-12, 13-17, 18-29, 30-49, 50-69, 70+). Los cortes son fijos, no se aprenden de los datos.

**Variables excluidas del modelo:** `SEMANA` (reemplazada por seno y coseno), `COD_MUN_O` y `MUNICIPIO_OCURRENCIA` (reemplazadas por `MUN_LOGCASOS`), `DEPARTAMENTO_OCURRENCIA` (duplica a `COD_DPTO_O`) y `MUN_CASOS` (se usa su logaritmo).

**Sobre `ANO`:** se incluye porque el split es aleatorio y el año captura la deriva de la tasa de hospitalización (53% en 2022 frente a ~35% en 2024-25, según el EDA). Limitación: el modelo no podría extrapolar a años futuros; una mejora sería un split temporal.

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, MinMaxScaler,
                                   StandardScaler, RobustScaler, PowerTransformer)

def preparar_categoricas(d):
    """Crea GRUPO_EDAD (cortes fijos) y deja las categóricas como texto simple."""
    d = d.copy()
    d["GRUPO_EDAD"] = pd.cut(
        d["EDAD_ANIOS"],
        bins=[0, 1, 5, 13, 18, 30, 50, 70, 200],
        labels=["menor_1", "1_4", "5_12", "13_17", "18_29", "30_49", "50_69", "70_mas"],
        right=False,
    ).astype(str)
    for c in ["SEXO", "TIP_SS", "GRUPO_EDAD"]:
        d[c] = d[c].astype(object)
    return d

train_d = preparar_categoricas(train_d)
test_d = preparar_categoricas(test_d)

def construir_preprocesador():
    """ColumnTransformer reutilizable: se ajusta (fit) solo con train."""
    return ColumnTransformer(
        transformers=[
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
             ["SEXO", "TIP_SS", "AREA", "PER_ETN", "TIP_CAS", "COD_DPTO_O", "GRUPO_EDAD"]),
            ("ordinal", Pipeline([
                ("orden", OrdinalEncoder(categories=[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]])),
                ("minmax", MinMaxScaler())]), ["ESTRATO"]),
            ("anio", MinMaxScaler(), ["ANO"]),
            ("edad", Pipeline([
                ("yeo", PowerTransformer(method="yeo-johnson", standardize=False)),
                ("robusto", RobustScaler())]), ["EDAD_ANIOS"]),
            ("mun", StandardScaler(), ["MUN_LOGCASOS"]),
            ("pasan", "passthrough", ["GP_DESPLAZ", "GP_MIGRANT", "GP_GESTAN", "SEMANA_SIN", "SEMANA_COS"]),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

X_train, y_train = train_d.drop(columns="PAC_HOS"), train_d["PAC_HOS"]
X_test, y_test = test_d.drop(columns="PAC_HOS"), test_d["PAC_HOS"]

prep = construir_preprocesador()
Xt_train = prep.fit_transform(X_train)   # fit SOLO con train
Xt_test = prep.transform(X_test)         # test solo se transforma

print("Forma train:", Xt_train.shape, "| test:", Xt_test.shape)
print("Nulos -> train:", int(Xt_train.isna().sum().sum()), "| test:", int(Xt_test.isna().sum().sum()))
print("\nColumnas finales:")
for i in range(0, len(Xt_train.columns), 5):
    print("  ", ", ".join(Xt_train.columns[i:i + 5]))

Forma train: (464856, 63) | test: (116215, 63)
Nulos -> train: 0 | test: 0

Columnas finales:
   SEXO_M, TIP_SS_E, TIP_SS_I, TIP_SS_N, TIP_SS_P
   TIP_SS_S, AREA_2, AREA_3, PER_ETN_2, PER_ETN_3
   PER_ETN_4, PER_ETN_5, PER_ETN_6, TIP_CAS_3, TIP_CAS_5
   COD_DPTO_O_5, COD_DPTO_O_8, COD_DPTO_O_13, COD_DPTO_O_15, COD_DPTO_O_17
   COD_DPTO_O_18, COD_DPTO_O_19, COD_DPTO_O_20, COD_DPTO_O_23, COD_DPTO_O_25
   COD_DPTO_O_27, COD_DPTO_O_41, COD_DPTO_O_44, COD_DPTO_O_47, COD_DPTO_O_50
   COD_DPTO_O_52, COD_DPTO_O_54, COD_DPTO_O_63, COD_DPTO_O_66, COD_DPTO_O_68
   COD_DPTO_O_70, COD_DPTO_O_73, COD_DPTO_O_76, COD_DPTO_O_81, COD_DPTO_O_85
   COD_DPTO_O_86, COD_DPTO_O_88, COD_DPTO_O_91, COD_DPTO_O_94, COD_DPTO_O_95
   COD_DPTO_O_97, COD_DPTO_O_99, GRUPO_EDAD_18_29, GRUPO_EDAD_1_4, GRUPO_EDAD_30_49
   GRUPO_EDAD_50_69, GRUPO_EDAD_5_12, GRUPO_EDAD_70_mas, GRUPO_EDAD_menor_1, ESTRATO
   ANO, EDAD_ANIOS, MUN_LOGCASOS, GP_DESPLAZ, GP_MIGRANT
   GP_GESTAN, SEMANA_SIN, SEMANA_COS


## 11. Verificación anti-leakage
El enunciado prohíbe normalizar antes del split. Esta celda lo comprueba con evidencia: los escaladores solo vieron las filas de train, y los parámetros que aprendieron coinciden con los de train y no con los del dataset completo.

In [19]:
sc_mun = prep.named_transformers_["mun"]
sc_anio = prep.named_transformers_["anio"]

print("Filas de train:", len(train_d), "| filas totales:", len(df_split))
print("Filas vistas por los escaladores:", sc_mun.n_samples_seen_, "y", sc_anio.n_samples_seen_)
assert sc_mun.n_samples_seen_ == len(train_d) == sc_anio.n_samples_seen_
assert np.isclose(sc_mun.mean_[0], train_d["MUN_LOGCASOS"].mean())

media_total = pd.concat([train_d, test_d])["MUN_LOGCASOS"].mean()
print("\nMedia de MUN_LOGCASOS que usa el escalador (train):", round(sc_mun.mean_[0], 5))
print("Media con train+test (NO usada)                  :", round(media_total, 5))
print("Media tras escalar -> train:", round(Xt_train["MUN_LOGCASOS"].mean(), 4),
      "| test:", round(Xt_test["MUN_LOGCASOS"].mean(), 4))
print("\nVerificación anti-leakage superada: el escalador no vio filas de test.")

Filas de train: 464856 | filas totales: 581071
Filas vistas por los escaladores: 464856.0 y 464856

Media de MUN_LOGCASOS que usa el escalador (train): 7.75782
Media con train+test (NO usada)                  : 7.75588
Media tras escalar -> train: -0.0 | test: -0.0057

Verificación anti-leakage superada: el escalador no vio filas de test.


## 12. Guardado de `datos_limpios.parquet`
Se guarda la matriz ya codificada y escalada (train y test juntos) con dos columnas extra: `PAC_HOS` (objetivo, 1 = hospitalizado) y `split` (`train`/`test`). Las Personas 4 y 5 deben **leer la columna `split`** en lugar de volver a dividir los datos, para que todo el equipo use la misma partición. Uso: `df = pd.read_parquet("datos_limpios.parquet")`, luego `train = df[df["split"] == "train"]` y `X = train.drop(columns=["PAC_HOS", "split"])`.

In [20]:
datos_limpios = pd.concat([
    Xt_train.assign(PAC_HOS=y_train.to_numpy(), split="train"),
    Xt_test.assign(PAC_HOS=y_test.to_numpy(), split="test"),
]).sort_index()

columnas_x = list(Xt_train.columns)
datos_limpios[columnas_x] = datos_limpios[columnas_x].astype("float32")
datos_limpios["PAC_HOS"] = datos_limpios["PAC_HOS"].astype("int8")

datos_limpios.to_parquet("datos_limpios.parquet", index=False)   # queda en entrega/

# Verificación: se vuelve a leer el archivo guardado
chk = pd.read_parquet("datos_limpios.parquet")
print("Forma:", chk.shape, "| columnas de entrada:", len(chk.columns) - 2)
print("Nulos:", int(chk.isna().sum().sum()))
print(chk["split"].value_counts())
print(chk.groupby("split")["PAC_HOS"].mean().round(4))

Forma: (581071, 65) | columnas de entrada: 63
Nulos: 0
split
train    464856
test     116215
Name: count, dtype: int64
split
test     0.3844
train    0.3844
Name: PAC_HOS, dtype: float64


## Resumen de la Fase B
1. **Diagnóstico:** nulos disfrazados en `estrato`, edades en unidades mezcladas, outliers, duplicados y variables posteriores a la notificación.
2. **Limpieza determinística (`limpiar_basico`)** y decisiones justificadas: conservar duplicados, conservar los outliers de edad (67-100 años) y excluir `AJUSTE` y `nom_est_f_caso`.
3. **Split** 80/20 estratificado **antes** de imputar y escalar.
4. **Imputación** ajustada con train: KNN para `ESTRATO` (mejor que la moda) y mediana para `EDAD_ANIOS`.
5. **Transformación y codificación:** Yeo-Johnson para la edad, log para el municipio, One-Hot y ordinal, y escalado (Robust, Z-score o Min-Max según la variable).
6. **Verificación anti-leakage** y archivo `datos_limpios.parquet` con las columnas `PAC_HOS` y `split`.